In [ ]:
import os,sys,json,hashlib,zipfile,io,subprocess,shutil,time
from pathlib import Path
from datetime import datetime,timezone
RUN_DIR=Path('/kaggle/working/cure_gpu_mechanism_01')/'cpu'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
RUN_DIR.mkdir(parents=True,exist_ok=False)
SOURCE=RUN_DIR/'source'
payload=__import__('base64').b64decode('...')
assert hashlib.sha256(payload).hexdigest()=='535dc4a0aa392503d5d80c9472cb82b312e37c33ba9b3a48c4396312d6fdeaaa'
with zipfile.ZipFile(io.BytesIO(payload)) as z:
    assert z.testzip() is None
    z.extractall(SOURCE)
sys.path.insert(0,str(SOURCE))
PLAN=json.loads((SOURCE/'study/protocol.json').read_text())
WHEELHOUSE_DIR=next(Path('/kaggle/input').rglob('swegemma-0.2.11-py3-none-any.whl')).parent
actual={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in WHEELHOUSE_DIR.glob('*.whl') if 'cutlass' not in p.name.lower()}
assert actual==PLAN['wheel_hashes'],'Wheelhouse bytes changed; stop before inference'
(RUN_DIR/'wheel_hashes.json').write_text(json.dumps(actual,indent=2))


In [ ]:
import glob
os.environ.update(TRANSFORMERS_NO_TF='1',TRANSFORMERS_NO_FLAX='1',TRANSFORMERS_OFFLINE='1',HF_HUB_OFFLINE='1',LITELLM_LOCAL_MODEL_COST_MAP='True',LITELLM_TELEMETRY='False',OTEL_SDK_DISABLED='true',PYTHONIOENCODING='utf-8')
for pattern in ('/usr/local/lib/python*/dist-packages/*cutlass*.pth','/usr/local/lib/python*/site-packages/*cutlass*.pth'):
    for path in glob.glob(pattern): Path(path).unlink(missing_ok=True)
wheels=list(Path('/kaggle/input').rglob('swegemma-0.2.11-py3-none-any.whl'))
assert len(wheels)==1
wheelhouse=wheels[0].parent
links=RUN_DIR/'wheelhouse_links'; links.mkdir()
for p in wheelhouse.glob('*.whl'):
    if 'cutlass' in p.name.lower(): continue
    name=p.name.replace('cu128','+cu128') if 'cu128' in p.name and '+' not in p.name else p.name
    (links/name).symlink_to(p)
pins=['swegemma-0.2.11-py3-none-any.whl','adk_submission-0.2.13-py3-none-any.whl','adk_eval_core-0.1.0-py3-none-any.whl','google_adk-1.36.1-py3-none-any.whl','google_genai-2.11.0-py3-none-any.whl']
try:
    with (RUN_DIR/'installation.log').open('x') as log:
        # Match official Getting Started: curated wheels, no dependency resolver.
        subprocess.run([sys.executable,'-m','pip','install','--no-index','--no-deps','--force-reinstall']+
            sorted(str(p) for p in links.glob('*.whl')),stdout=log,stderr=subprocess.STDOUT,check=True,timeout=1800)
finally:
    shutil.rmtree(links)


In [ ]:
try:
    for label,args in [('original63',['-m','cure.cpu_validation','cpu-r03',str(RUN_DIR/'validation')]),('study',['-m','unittest','cure.gpu_mechanism.test_study','cure.gpu_mechanism.test_official_study','-v'])]:
        p=subprocess.run([sys.executable,*args],cwd=SOURCE,capture_output=True,text=True,timeout=1200)
        (RUN_DIR/(label+'.log')).write_text(p.stdout+p.stderr)
        print(label,p.stdout+p.stderr,flush=True)
        assert p.returncode==0,label
    (RUN_DIR/'study_cpu_gate.json').write_text(json.dumps({'passed':True,'gpu_inference':False}))
finally:
    from harness_v2.evidence import bundle
    print('CPU_STUDY_EVIDENCE',bundle(RUN_DIR),flush=True)
